# MICO Recipe Grouping 조회
SQLite DB에서 RecipeGroup 데이터를 Product / Oper Desc 기준으로 조회

In [1]:
import sqlite3
import pandas as pd

DB_PATH = '../db.sqlite3'

conn = sqlite3.connect(DB_PATH)
print('DB 연결 완료')

DB 연결 완료


## 1. 전체 Recipe Group 목록 (Category 정보 포함)

In [2]:
df_groups = pd.read_sql_query("""
    SELECT
        rg.id          AS group_id,
        rg.name        AS group_name,
        c.product,
        c.oper_id,
        c.oper_desc,
        rg.created_at
    FROM setup_mico_recipegroup AS rg
    JOIN setup_mico_category    AS c  ON rg.category_id = c.id
    ORDER BY c.product, c.oper_desc, rg.name
""", conn)

print(f'전체 Recipe Group: {len(df_groups)}건')
df_groups

전체 Recipe Group: 1건


,group_id,group_name,product,oper_id,oper_desc,created_at
0,1,LC_SNBPSG,LC,A908740A,SN BPSG CMP,2026-03-17 09:44:09.338361


## 2. Recipe Group + 소속 SubCategory (전체 상세)

In [3]:
df_full = pd.read_sql_query("""
    SELECT
        c.product,
        c.oper_id,
        c.oper_desc,
        rg.name        AS group_name,
        s.fab,
        s.device,
        s.recipe_id,
        s.maker
    FROM setup_mico_recipegroup             AS rg
    JOIN setup_mico_category                AS c   ON rg.category_id    = c.id
    JOIN setup_mico_recipegroup_subcategories AS m  ON m.recipegroup_id  = rg.id
    JOIN setup_mico_subcategory             AS s   ON m.subcategory_id  = s.id
    ORDER BY c.product, c.oper_desc, rg.name, s.fab, s.device
""", conn)

print(f'전체 Recipe Group - SubCategory 매핑: {len(df_full)}건')
df_full

전체 Recipe Group - SubCategory 매핑: 2건


,product,oper_id,oper_desc,group_name,fab,device,recipe_id,maker
0,LC,A908740A,SN BPSG CMP,LC_SNBPSG,M14,E2,E2_SNBPSG_R03_AB,EBARA
1,LC,A908740A,SN BPSG CMP,LC_SNBPSG,M14,E9,E9_SNBPSG_R03_AB,EBARA


## 3. Product / Oper Desc 기준 필터링

In [4]:
# ← 원하는 값으로 변경 (빈 문자열 '' 로 두면 전체 조회)
TARGET_PRODUCT   = 'LC'
TARGET_OPER_DESC = ''      # 예: 'M1 CU CMP'

mask = pd.Series([True] * len(df_full))

if TARGET_PRODUCT:
    mask &= df_full['product'] == TARGET_PRODUCT

if TARGET_OPER_DESC:
    mask &= df_full['oper_desc'].str.contains(TARGET_OPER_DESC, case=False, na=False)

df_filtered = df_full[mask].reset_index(drop=True)
print(f'필터 결과: {len(df_filtered)}건  (product={TARGET_PRODUCT or "전체"}, oper_desc={TARGET_OPER_DESC or "전체"})')
df_filtered

필터 결과: 2건  (product=LC, oper_desc=전체)


,product,oper_id,oper_desc,group_name,fab,device,recipe_id,maker
0,LC,A908740A,SN BPSG CMP,LC_SNBPSG,M14,E2,E2_SNBPSG_R03_AB,EBARA
1,LC,A908740A,SN BPSG CMP,LC_SNBPSG,M14,E9,E9_SNBPSG_R03_AB,EBARA


## 4. SQL 직접 필터 버전 (대용량 데이터 대비)

In [5]:
TARGET_PRODUCT   = 'LC'
TARGET_OPER_DESC = 'M1 CU CMP'   # 부분 일치 검색 (LIKE)

query = """
    SELECT
        c.product,
        c.oper_id,
        c.oper_desc,
        rg.name        AS group_name,
        s.fab,
        s.device,
        s.recipe_id,
        s.maker
    FROM setup_mico_recipegroup               AS rg
    JOIN setup_mico_category                  AS c   ON rg.category_id   = c.id
    JOIN setup_mico_recipegroup_subcategories  AS m   ON m.recipegroup_id = rg.id
    JOIN setup_mico_subcategory               AS s   ON m.subcategory_id = s.id
    WHERE c.product   = ?
      AND c.oper_desc LIKE ?
    ORDER BY rg.name, s.fab, s.device
"""

df_sql = pd.read_sql_query(query, conn, params=(TARGET_PRODUCT, f'%{TARGET_OPER_DESC}%'))
print(f'SQL 필터 결과: {len(df_sql)}건')
df_sql

SQL 필터 결과: 0건


,product,oper_id,oper_desc,group_name,fab,device,recipe_id,maker


## 5. 그룹별 요약 (그룹 당 SubCategory 수)

In [6]:
df_summary = pd.read_sql_query("""
    SELECT
        c.product,
        c.oper_desc,
        rg.name             AS group_name,
        COUNT(m.subcategory_id) AS subcategory_count
    FROM setup_mico_recipegroup               AS rg
    JOIN setup_mico_category                  AS c   ON rg.category_id   = c.id
    LEFT JOIN setup_mico_recipegroup_subcategories AS m ON m.recipegroup_id = rg.id
    GROUP BY rg.id, c.product, c.oper_desc, rg.name
    ORDER BY c.product, c.oper_desc, rg.name
""", conn)

print(f'그룹 요약: {len(df_summary)}건')
df_summary

그룹 요약: 1건


,product,oper_desc,group_name,subcategory_count
0,LC,SN BPSG CMP,LC_SNBPSG,2


## 6. 사용 가능한 Product / Oper Desc 목록 확인

In [7]:
df_options = pd.read_sql_query("""
    SELECT DISTINCT c.product, c.oper_desc
    FROM setup_mico_recipegroup AS rg
    JOIN setup_mico_category    AS c ON rg.category_id = c.id
    ORDER BY c.product, c.oper_desc
""", conn)

print('Recipe Group 이 등록된 Product / Oper Desc 목록:')
df_options

Recipe Group 이 등록된 Product / Oper Desc 목록:


,product,oper_desc
0,LC,SN BPSG CMP


In [8]:
conn.close()
print('DB 연결 종료')

DB 연결 종료
